# Time Series Battle Notebook

Univariate XGBoost kit (`tskit`): prepare → stationarity → CV → fit → predict → submission.

On contest day: point `TRAIN_PATH` / `TEST_PATH` at the official files and set `date_col` / `target_col` / `exog_cols` / `freq` / `horizon`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tskit import (
    TSConfig,
    prepare_train_test,
    check_stationarity,
    walk_forward_validate,
    summarize_cv,
    XGBForecaster,
    to_submission,
)

TRAIN_PATH = ROOT / "data" / "synthetic" / "train.csv"
TEST_PATH = ROOT / "data" / "synthetic" / "test.csv"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_raw.head()


In [ ]:
cfg = TSConfig(
    date_col="date",
    target_col="y",
    freq="D",
    horizon=14,
    exog_cols=("promo",),  # known-in-future covariates; use () if none
    lags=(1, 7, 14),
    rolling_windows=(7, 14),
    target_diff=True,
    strategy="recursive",  # or "direct"
    n_splits=3,
    gap=0,
    metric="mae",
)
cfg


## Prepare (reindex + NaN policy)

In [ ]:
train, test = prepare_train_test(train_raw, test_raw, cfg)
print(train.shape, test.shape)
train.tail(3)


## Stationarity diagnostics (ADF / KPSS)
Diagnostic only — does not block training. Differencing for the model is controlled by `cfg.target_diff`.

In [ ]:
stationarity = check_stationarity(train, cfg)
stationarity[
    [
        "series",
        "adf_pvalue",
        "adf_stationary_at_5pct",
        "kpss_pvalue",
        "kpss_stationary_at_5pct",
    ]
]


## Walk-forward CV (compare recursive vs direct)

In [ ]:
cv_results = walk_forward_validate(
    train,
    cfg,
    strategies=["recursive", "direct"],
)
summary = summarize_cv(cv_results)
display(summary)

best_strategy = summary.sort_values("mean_score").iloc[0]["strategy"]
print("best strategy by CV:", best_strategy)
display(cv_results[best_strategy]["per_horizon"])


## Fit + predict
Uses the CV-selected strategy. Optionally run `model.tune(train, cfg, n_trials=40)` before the final fit.

In [ ]:
cfg_final = cfg.with_updates(strategy=best_strategy)
model = XGBForecaster(cfg_final)

# Optional short HPO (commented for a fast first submit):
# best_params = model.tune(train, cfg_final, n_trials=40)
# print(best_params)

model.fit(train, cfg_final)
preds = model.predict(train, test, cfg_final)
preds.head()


## Submission

In [ ]:
sub_path = OUT_DIR / "submission.csv"
submission = to_submission(
    test_raw,
    preds,
    sub_path,
    date_col=cfg.date_col,
    pred_col="prediction",
    # id_col="id",  # uncomment if the contest wants an id column
)
print("wrote", sub_path)
submission.head()


## Contest-day checklist
1. Set paths + `date_col` / `target_col` / `freq` / `horizon` / `exog_cols`.
2. Confirm metric (`mae` / `rmse` / `rmsle` / `mape`) matches the brief if known.
3. Run CV; pick strategy; if trend is weak try `target_diff=False`.
4. Optional `tune()`; then fit → predict → submit.
5. Align `to_submission` columns with the official format (`id` vs `date` vs prediction-only).
